# CHELSA V2.1 — normal climática do Brasil (1991-2020)

Baixa e recorta para o Brasil (bbox retangular) as variáveis mensais `tas`, `pet` e `pr` do
[CHELSA V2.1](https://chelsa-climate.org/), calcula a normal de 30 anos (1991-2020) por mês e
monta uma imagem GeoTIFF multibanda por variável (12 bandas cada: um mês por banda), pronta
para subir manualmente como **3 assets separados** no Google Earth Engine (um para `tas`, um
para `pet` e um para `pr`).

PET tem meses faltantes na fonte; a média usa apenas os anos disponíveis em cada mês.


## 1. Baixar e recortar os arquivos mensais

Gera o manifesto de disponibilidade e recorta cada arquivo mensal para o bbox do Brasil (sem baixar os GeoTIFFs globais inteiros).

In [ ]:
import sys
sys.path.insert(0, ".")

import baixar_recortar
baixar_recortar.main()

## 2. Conferir o manifesto de disponibilidade

In [ ]:
import pandas as pd

manifesto = pd.read_csv(baixar_recortar.MANIFESTO_PATH)
manifesto.groupby("variavel")["encontrado"].value_counts().unstack(fill_value=0)

## 3. Calcular a normal 1991-2020 e montar a imagem multibanda

In [ ]:
import gerar_normal_multibanda
gerar_normal_multibanda.main()

## 4. Conferência visual de uma banda por variável

In [ ]:
import rasterio
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 3, figsize=(15, 5))
for eixo, var in zip(eixos, gerar_normal_multibanda.VARIAVEIS):
    with rasterio.open(gerar_normal_multibanda.saida_path(var)) as src:
        dado = src.read(1)  # banda 1 = mês 01
        im = eixo.imshow(dado, cmap='viridis')
        eixo.set_title(src.descriptions[0])
        plt.colorbar(im, ax=eixo, shrink=0.7)
plt.tight_layout()
plt.show()


## 5. Checagem de sanidade da imagem final

In [ ]:
for var in gerar_normal_multibanda.VARIAVEIS:
    caminho = gerar_normal_multibanda.saida_path(var)
    if not caminho.exists():
        print(f'{var}: arquivo não encontrado ({caminho})')
        continue
    with rasterio.open(caminho) as src:
        print(f'=== {var} ===')
        print('Arquivo:', caminho)
        print('Bandas:', src.count)
        print('Dimensões:', src.width, 'x', src.height)
        print('Dtype:', src.dtypes[0])
        print('CRS:', src.crs)
        print('Nomes das bandas:', list(src.descriptions))
        print()


## 6. Upload manual para o Google Earth Engine

Este notebook não faz upload automático. Para subir cada imagem como asset separado:

1. Enviar os 3 arquivos de `dados_chelsa/normal_1991_2020/` para um bucket do Google Cloud Storage.
2. Rodar, um comando por variável:

```powershell
earthengine upload image --asset_id=projects/SEU_PROJETO/assets/chelsa_brasil_tas_normal_1991_2020 gs://SEU_BUCKET/chelsa_brasil_tas_normal_1991_2020.tif
earthengine upload image --asset_id=projects/SEU_PROJETO/assets/chelsa_brasil_pet_normal_1991_2020 gs://SEU_BUCKET/chelsa_brasil_pet_normal_1991_2020.tif
earthengine upload image --asset_id=projects/SEU_PROJETO/assets/chelsa_brasil_pr_normal_1991_2020 gs://SEU_BUCKET/chelsa_brasil_pr_normal_1991_2020.tif
```
